# Multi-Branch Ensemble (`2..5` Models: Hierarchical & End-to-End Multi-Model Blending)

This notebook builds a scalable **2-to-5 model ensemble** on top of `MultiBranchEnsemble` (`utils.py`) in the same style and search architecture as `multibranch.ipynb`:
- **Scalable Member Count (`n_models = 2..5`)**: Combine between 2 and 5 diverse architectures and sensor specialists (`all`-modality vs `imu`-only specialists for robustness when Thermopile/ToF are absent).
- **Two Flexible Sources (`use_saved_bundles`)**:
  - `use_saved_bundles = True`: Load saved model bundles (`weights.pt`, `config.json`, `oof_proba.csv`, `manifest.json`) exported by prior `multibranch.ipynb` runs (`BUNDLE_DIRS`) and learn zero-leakage OOF meta-weights / hierarchical routing in seconds.
  - `use_saved_bundles = False`: Train `n_models` distinct end-to-end models directly using `MODEL_SPECS`, compute GroupKFold OOF predictions across folds, optimize blend weights & routing threshold, and refit all `n_models` members.
- **Hierarchical or Direct End-to-End Blending (`ensemble_mode`)**:
  - `'hierarchical'`: Competition-aware Stage-1 binary `bfrb` vs `non_bfrb` gate (with OOF-tuned `binary_threshold` and modality-aware `imu_specialist_boost`) + Stage-2 specific gesture routing.
  - `'soft_vote'`: Modality-aware OOF-optimized or fixed weighted probability average.
  - `'stacking'`: Regularized logistic meta-learner over member probabilities + THM/ToF availability flags.


In [ ]:
import os
import sys
import json
import warnings
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

warnings.filterwarnings('ignore')

# Local path routing
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Robust Kaggle path routing (handles nested folders and symlinks from zip uploads)
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
        if 'utils.py' in files or 'base_utils.py' in files or 'pretraining_ssl_utils.py' in files:
            if root not in sys.path:
                sys.path.insert(0, root)

sys.path.insert(0, src_path)

from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline

try:
    import skopt
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

import base_utils
import pretraining_ssl_utils
import utils
importlib.reload(utils)
from utils import (
    DEFAULT_ENSEMBLE_MODEL_SPECS,
    Ensemble,
    MultiBranchEnsemble,
    MultiBranchTSAIClassifier,
    ECat,
    EInt,
    EFloat,
    GASearchCV,
    HierarchicalGASearchCV,
    HierarchicalBayesSearchCV,
    build_quick_results_df,
    evaluate_holdout,
    prepare_bayesian_space,
    prepare_evolutionary_space,
    candidate_score,
    deduplicate_param_grid,
    run_validation_searches,
    find_data_root,
    library_status,
)

print('Ensemble imports loaded successfully')
print('Library availability:', library_status())
print('CUDA available:', torch.cuda.is_available(), '| Device count:', torch.cuda.device_count())
warnings.filterwarnings('always', message='Ensemble candidate skipped:', category=RuntimeWarning)


# Setup

In [ ]:
experiment_name = 'multibranch_ensemble_bfrb'

# Supported targets: 'bfrb', 'gesture_action', 'orientation', 'gesture', 'gesture_position'
TARGET_COL = 'bfrb'

# Use sample.csv for a quick smoke test; set False for full train.csv
use_sample_data = False
sample_file = 'sample.csv'

# Ensemble scaling (2 to 5 models) & source mode
n_models = 3                   # Set 2, 3, 4, or 5
use_saved_bundles = False      # True: blend saved multibranch.ipynb bundles; False: train 2..5 E2E models

# Paths to saved bundles from multibranch.ipynb (used when use_saved_bundles=True)
BUNDLE_DIRS = [
    # str(Path(workspace_root) / 'results_multibranch_bfrb_grid' / 'model_1_bundle'),
    # str(Path(workspace_root) / 'results_multibranch_bfrb_grid' / 'model_2_bundle'),
    # str(Path(workspace_root) / 'results_multibranch_bfrb_grid' / 'model_3_bundle'),
]

# Diverse member specs for end-to-end training (sliced to [:n_models])
MODEL_SPECS = [
    {'name': 'multimodal_inception', 'arch': 'InceptionTimePlus', 'extractor__modalities': 'all', 'fusion_type': 'attention'},
    {'name': 'imu_specialist_conv', 'arch': 'Conv1d', 'extractor__modalities': 'imu', 'fusion_type': 'concat'},
    {'name': 'multimodal_resnet', 'arch': 'ResNetPlus', 'extractor__modalities': 'all', 'fusion_type': 'gated'},
    {'name': 'imu_specialist_resnet', 'arch': 'ResNetPlus', 'extractor__modalities': 'imu', 'fusion_type': 'concat'},
    {'name': 'multimodal_fcn', 'arch': 'FCNPlus', 'extractor__modalities': 'all', 'fusion_type': 'attention'},
]

# Shared base training hyperparameters for end-to-end members (0 = no pretraining/SSL)
BASE_ESTIMATOR_PARAMS = {
    'epochs': 20,
    'batch_size': 64,
    'learning_rate': 1e-3,
    'branch_filters': 64,
    'branch_units': 96,
    'fusion_units': 96,
    'dense_units': 96,
    'early_stopping_patience': 6,
    'pretraining_mode': 'none',
    'pretraining_epochs': 0,
    'finetuning_head_epochs': 0,
    'ssl_mask_ratio': 0.0,
    'ssl_span_length': 0,
    'ssl_decoder_units': 0,
}

search_mode = 'grid'           # Choose 'bayes', 'evo', or 'grid'
random_state = 42
n_splits = 2                   # Outer CV splits for ensemble search (1 -> GroupShuffleSplit; >=2 -> GroupKFold)
ensemble_cv_splits = 2         # Inner GroupKFold splits per E2E member for leak-free OOF generation

train_size = 0.8
n_iter = 6
verbose = 3
error_score = -1.0
n_jobs = 1
refit = True

# Evolutionary search controls
population_size = 8
generations = 3
tournament_size = 3
elitism = True
crossover_probability = 0.7
mutation_probability = 0.15

INVALID_CANDIDATE_POLICY = 'penalize'

orientation_filter_list = []
gesture_position_filter_list = []
gesture_filter_list = []
target_only_bool = False

simulate_hidden_test = False
hidden_test_null_fraction = 0.50
save_ensemble_bundle = True

search_mode = str(search_mode).strip().lower()
if search_mode not in ('evo', 'bayes', 'grid'):
    raise ValueError("search_mode must be 'evo', 'bayes', or 'grid'")

results_dir = Path(workspace_root) / f'results_{experiment_name}_{TARGET_COL}_{search_mode}'
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=(1.0 - train_size),
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)

VALIDATION_PARAM_SPACE = {
    'validation_mode': ['sequence'],
    'group_column': ['subject'],
}


# Prep

In [ ]:
# Load and prepare training data without retaining a second sensor frame.
data_root = find_data_root()
sample_path = data_root / sample_file
if not sample_path.exists():
    sample_path = data_root / 'sample.csv'
train_path = sample_path if (use_sample_data and sample_path.exists()) else (data_root / 'train.csv')
dem_path = data_root / 'train_demographics.csv'

train_df = pd.read_csv(train_path)
if dem_path.exists():
    dem_df = pd.read_csv(dem_path)
    new_dem_cols = ['subject'] + [c for c in dem_df.columns if c not in train_df.columns]
    train_df = train_df.merge(dem_df[new_dem_cols], on='subject', how='left')
    del dem_df

print('Data root:', data_root)
print('Loaded training file:', train_path.name, '| rows:', len(train_df), '| seqs:', train_df['sequence_id'].nunique())

sensor_columns = [
    c for c in train_df.columns
    if c.startswith(('acc_', 'rot_', 'thm_', 'tof_'))
]
print(f'Detected {len(sensor_columns)} raw sensor columns')

if orientation_filter_list and 'orientation' in train_df.columns:
    train_df = train_df[train_df['orientation'].isin(orientation_filter_list)]
if gesture_position_filter_list and 'gesture_position' in train_df.columns:
    train_df = train_df[train_df['gesture_position'].isin(gesture_position_filter_list)]
if gesture_filter_list and 'gesture' in train_df.columns:
    train_df = train_df[train_df['gesture'].isin(gesture_filter_list)]
if target_only_bool and 'sequence_type' in train_df.columns:
    train_df = train_df[train_df['sequence_type'] == 'Target']

if 'bfrb' not in train_df.columns:
    is_tgt = (train_df['sequence_type'] == 'Target') if 'sequence_type' in train_df.columns else pd.Series(True, index=train_df.index)
    train_df['bfrb'] = np.where(is_tgt, train_df['gesture'].astype(str), 'non_bfrb')

experiment_df = train_df.reset_index(drop=True)
del train_df
print('Available sequences:', experiment_df['sequence_id'].nunique())


In [ ]:
ensemble_estimator = MultiBranchEnsemble(
    bundle_dirs=BUNDLE_DIRS if use_saved_bundles else None,
    model_specs=MODEL_SPECS,
    base_estimator_params=BASE_ESTIMATOR_PARAMS,
    n_models=n_models,
    use_saved_bundles=use_saved_bundles,
    ensemble_mode='hierarchical',
    model_weights_mode='oof_opt',
    binary_threshold='auto',
    imu_specialist_boost=0.25,
    meta_C=1.0,
    cv_splits=ensemble_cv_splits,
    base_epochs=BASE_ESTIMATOR_PARAMS.get('epochs', 20),
    base_batch_size=BASE_ESTIMATOR_PARAMS.get('batch_size', 64),
    primary_target=TARGET_COL,
    validation_group_column='subject',
    invalid_candidate_policy=INVALID_CANDIDATE_POLICY,
    invalid_candidate_score=error_score,
    verbose=verbose,
    random_state=random_state,
)
pipeline = Pipeline([('estimator', ensemble_estimator)])


# Params

Separate `GRID_PARAM_SPACE`, `BAYESIAN_PARAM_SPACE`, and `EVOLUTIONARY_PARAM_SPACE` covering ensemble scaling (`n_models = 2..5`), hierarchical Stage-1 gating, and stacking/soft-vote controls with zero `.update()` calls.


## Grid

In [ ]:
# GRID SEARCH ONLY
GRID_PARAM_SPACE = {
    'estimator__n_models': [n_models],
    'estimator__use_saved_bundles': [use_saved_bundles],
    'estimator__ensemble_mode': ['hierarchical', 'soft_vote', 'stacking'],
    'estimator__model_weights_mode': ['oof_opt', 'equal'],
    'estimator__model_weights': [None],
    'estimator__binary_threshold': ['auto'],
    'estimator__imu_specialist_boost': [0.0, 0.25, 0.5],
    'estimator__meta_C': [1.0],
    'estimator__cv_splits': [ensemble_cv_splits],
    'estimator__base_epochs': [BASE_ESTIMATOR_PARAMS.get('epochs', 20)],
    'estimator__base_batch_size': [BASE_ESTIMATOR_PARAMS.get('batch_size', 64)],
    'estimator__primary_target': [TARGET_COL],
    'estimator__validation_group_column': ['subject'],
    'estimator__sequence_col': ['sequence_id'],
    'estimator__invalid_candidate_policy': [INVALID_CANDIDATE_POLICY],
    'estimator__invalid_candidate_score': [error_score],
    'estimator__device': ['auto'],
    'estimator__verbose': [verbose],
    'estimator__random_state': [random_state],
}


## Bayesian

In [ ]:
# BAYESIAN SEARCH ONLY
BAYESIAN_PARAM_SPACE = dict()
if SKOPT_AVAILABLE:
    BAYESIAN_PARAM_SPACE = {
        'estimator__n_models': Categorical([n_models]),
        'estimator__use_saved_bundles': Categorical([use_saved_bundles]),
        'estimator__ensemble_mode': Categorical(['hierarchical', 'soft_vote', 'stacking']),
        'estimator__model_weights_mode': Categorical(['oof_opt', 'equal']),
        'estimator__model_weights': Categorical([None]),
        'estimator__binary_threshold': Categorical(['auto']),
        'estimator__imu_specialist_boost': Real(0.0, 1.0),
        'estimator__meta_C': Real(0.1, 5.0, prior='log-uniform'),
        'estimator__cv_splits': Categorical([ensemble_cv_splits]),
        'estimator__base_epochs': Categorical([BASE_ESTIMATOR_PARAMS.get('epochs', 20)]),
        'estimator__base_batch_size': Categorical([BASE_ESTIMATOR_PARAMS.get('batch_size', 64)]),
        'estimator__primary_target': Categorical([TARGET_COL]),
        'estimator__validation_group_column': Categorical(['subject']),
        'estimator__sequence_col': Categorical(['sequence_id']),
        'estimator__invalid_candidate_policy': Categorical([INVALID_CANDIDATE_POLICY]),
        'estimator__invalid_candidate_score': Categorical([error_score]),
        'estimator__device': Categorical(['auto']),
        'estimator__verbose': Categorical([verbose]),
        'estimator__random_state': Categorical([random_state]),
    }
    BAYESIAN_PARAM_SPACE = prepare_bayesian_space(BAYESIAN_PARAM_SPACE)


## Evolution

In [ ]:
# EVOLUTIONARY SEARCH ONLY
EVOLUTIONARY_PARAM_SPACE = {
    'estimator__n_models': ECat([n_models]),
    'estimator__use_saved_bundles': ECat([use_saved_bundles]),
    'estimator__ensemble_mode': ECat(['hierarchical', 'soft_vote', 'stacking']),
    'estimator__model_weights_mode': ECat(['oof_opt', 'equal']),
    'estimator__model_weights': ECat([None]),
    'estimator__binary_threshold': ECat(['auto']),
    'estimator__imu_specialist_boost': EFloat(0.0, 1.0),
    'estimator__meta_C': EFloat(0.1, 5.0, prior='log-uniform'),
    'estimator__cv_splits': ECat([ensemble_cv_splits]),
    'estimator__base_epochs': ECat([BASE_ESTIMATOR_PARAMS.get('epochs', 20)]),
    'estimator__base_batch_size': ECat([BASE_ESTIMATOR_PARAMS.get('batch_size', 64)]),
    'estimator__primary_target': ECat([TARGET_COL]),
    'estimator__validation_group_column': ECat(['subject']),
    'estimator__sequence_col': ECat(['sequence_id']),
    'estimator__invalid_candidate_policy': ECat([INVALID_CANDIDATE_POLICY]),
    'estimator__invalid_candidate_score': ECat([error_score]),
    'estimator__device': ECat(['auto']),
    'estimator__verbose': ECat([verbose]),
    'estimator__random_state': ECat([random_state]),
}
EVOLUTIONARY_PARAM_SPACE = prepare_evolutionary_space(EVOLUTIONARY_PARAM_SPACE)


# Train

In [ ]:
if search_mode == 'bayes':
    if not SKOPT_AVAILABLE:
        raise ImportError('scikit-optimize is required when search_mode == "bayes"')
    param_space = BAYESIAN_PARAM_SPACE
    search = HierarchicalBayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        cv=cv_object,
        scoring=candidate_score,
        refit=refit,
        random_state=random_state,
        verbose=verbose,
        error_score=error_score,
        n_jobs=n_jobs,
    )
elif search_mode == 'evo':
    param_space = EVOLUTIONARY_PARAM_SPACE
    search = HierarchicalGASearchCV(
        estimator=pipeline,
        param_grid=param_space,
        cv=cv_object,
        scoring=candidate_score,
        refit=refit,
        population_size=population_size,
        generations=generations,
        tournament_size=tournament_size,
        elitism=elitism,
        crossover_probability=crossover_probability,
        mutation_probability=mutation_probability,
        random_state=random_state,
        verbose=verbose,
        error_score=error_score,
        n_jobs=n_jobs,
    )
else:
    deduped_grid = deduplicate_param_grid(GRID_PARAM_SPACE, base_params=pipeline.get_params(deep=True))
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=deduped_grid,
        scoring=candidate_score,
        cv=cv_object,
        refit=refit,
        verbose=verbose,
        error_score=error_score,
        n_jobs=n_jobs,
    )

validation_runs = run_validation_searches(
    search=search,
    experiment_df=experiment_df,
    target_col=TARGET_COL,
    validation_param_space=VALIDATION_PARAM_SPACE,
    sensor_columns=sensor_columns,
    train_size=train_size,
    n_splits=n_splits,
    random_state=random_state,
    results_dir=results_dir,
    experiment_name=experiment_name,
    search_mode=search_mode,
    timestamp=timestamp,
    simulate_hidden_test=simulate_hidden_test,
    hidden_test_null_fraction=hidden_test_null_fraction,
    save_ensemble_bundle=save_ensemble_bundle,
)

last_run = validation_runs['last_run']
protocol_results = validation_runs['protocol_results']
experiment_succeeded = bool(last_run.get('experiment_succeeded', False))
if experiment_succeeded:
    search = last_run['search']
    X_train = last_run['X_train']
    y_train = last_run['y_train']
    X_test = last_run['X_test']
    y_test = last_run['y_test']
    eval_results = last_run['holdout_results']


# Analysis

In [ ]:
if experiment_succeeded:
    best_model = search.best_estimator_
    best_ensemble = best_model.named_steps['estimator']
    print('Ensemble mode:', best_ensemble.ensemble_mode, '| n_models:', len(best_ensemble.members_))
    print('Learned weights:', np.round(best_ensemble.model_weights_, 4).tolist())
    print('Stage-1 binary threshold:', round(float(best_ensemble.binary_threshold_), 4))
    print('Last validation protocol:', last_run['validation_mode'])
    if eval_results:
        print('Holdout competition score (normal):', eval_results.get('competition_score'))
        if 'hidden_test_competition_score' in eval_results:
            print('Holdout competition score (50% Thm/ToF nullified):', eval_results['hidden_test_competition_score'])
    member_df = best_ensemble.summarize_members()
    display(member_df)
    if plt is not None and len(member_df):
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.barh(member_df['member'], member_df['oof_score'], color='steelblue')
        ax.set_xlabel('OOF Score')
        ax.set_title('Base Members vs Ensemble OOF Score')
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()
    print('Experiments CSV:', validation_runs['experiments_path'])
    print('Summary JSON:', validation_runs['summary_path'])
    print('Ensemble bundle dir:', validation_runs.get('bundle_dir'))
